# Network Event Threat Triage

Heuristic detection and a supervised baseline on flow-like logs (`src_ip`, `dst_ip`, `port`, `bytes`).

**What this notebook does**
1. Scores each event with simple source/destination behavior flags (ports, fan-out, bytes, shared targets).
2. Aggregates per source IP and labels likely **port scan**, **network scan**, **brute force**, **beaconing**, or **data exfiltration**.
3. Trains an XGBoost classifier on those labels as a first ML pass.

**How to read the model results.** Labels are rule-based. The model is trained on features that those rules already use, so a high test score mainly shows the pipeline can recover the heuristics — not that it would detect unseen attacks in production.

Data: `network_events.csv` in this folder.

In [ ]:
import pandas as pd
df = pd.read_csv("network_events.csv")

In [ ]:
port_counts = df.groupby("src_ip")["port"].nunique()
threshold = 2
suspicious_src = port_counts[port_counts > threshold].index
df["suspicious_try"] = df["src_ip"].isin(suspicious_src)


srcip = df.groupby("src_ip")["dst_ip"].nunique()
too_much = 2
suspicious_ip = srcip[srcip >= too_much].index
df["suspicious_ip"] = df["src_ip"].isin(suspicious_ip)


bytes_num = df.groupby("src_ip")["bytes"].sum()
too_muchh = 9000
suspicious_bytes = bytes_num[bytes_num > too_muchh].index
df["suspicious_bytes"] = df["src_ip"].isin(suspicious_bytes)


spread = df.groupby(["src_ip", "port"])["dst_ip"].nunique()
too_muchhh = 3
suspicious_pair = spread[spread >= too_muchhh].index
df["suspicious_spread"] = pd.Series(list(zip(df["src_ip"], df["port"]))).isin(suspicious_pair)


many_src = df.groupby("dst_ip")["src_ip"].nunique()
too_muchhhh = 3
suspicious_target = many_src[many_src >= too_muchhhh].index
df["suspicious_target"] = df["dst_ip"].isin(suspicious_target)


df["suspicious_score"] = (df["suspicious_try"].astype(int) + df["suspicious_ip"].astype(int) + df["suspicious_bytes"].astype(int) + df["suspicious_spread"].astype(int) + df["suspicious_target"].astype(int))

df["risk_level"] = "low"
df.loc[df["suspicious_score"] == 0, "risk_level"] = "normal"
df.loc[df["suspicious_score"] == 1, "risk_level"] = "low"
df.loc[df["suspicious_score"] == 2, "risk_level"] = "medium"
df.loc[df["suspicious_score"] == 3, "risk_level"] = "high"
df.loc[df["suspicious_score"] == 4, "risk_level"] = "critical"


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

df[["src_ip", "dst_ip", "bytes", "port", "suspicious_try", "suspicious_ip", "suspicious_bytes", "suspicious_spread", "suspicious_target", "suspicious_score", "risk_level",]]


In [ ]:
df["src_event_count"] = df.groupby("src_ip")["src_ip"].transform("count")
df["bytes_sum"] = df.groupby("src_ip")["bytes"].transform("sum")
df["avg_bytes_per_event"] = (df["bytes_sum"] // df["src_event_count"])
df["max_bytes_per_event"] = df.groupby("src_ip")["bytes"].transform("max")
df["unique_ports_per_src_ip"] = df.groupby("src_ip")["port"].transform("nunique")
df["unique_dst_count"] = df.groupby("src_ip")["dst_ip"].transform("nunique")

df[["src_ip", "bytes", "src_event_count", "bytes_sum", "avg_bytes_per_event", "max_bytes_per_event", "unique_ports_per_src_ip", "unique_dst_count"]].drop_duplicates(subset="src_ip")


In [ ]:
df["attack_type"] = "normal"


df.loc[
(df["attack_type"] == "normal") &
(df["max_bytes_per_event"] >= 8000) &
(df["bytes_sum"] >= 15000) &
(df["unique_dst_count"] <= 2),
"attack_type"
] = "data_exfiltration"


df.loc[
(df["attack_type"] == "normal") &
(df["unique_dst_count"] >= 4) &
(df["unique_ports_per_src_ip"] <= 2) &
(df["avg_bytes_per_event"] <= 500) &
(df["src_event_count"] >= 4),
"attack_type"
] = "network_scan"


df.loc[
(df["attack_type"] == "normal") &
(df["unique_ports_per_src_ip"] >= 6)&
(df["src_event_count"] >= 6)&
(df["avg_bytes_per_event"] <= 500)&
(df["unique_dst_count"] <= 2),
"attack_type"
] = "port_scan"


df.loc[
(df["attack_type"] == "normal") &
(df["src_event_count"] >= 6) &
(df["unique_dst_count"] == 1) &
(df["unique_ports_per_src_ip"] == 1)&
(df["avg_bytes_per_event"] >= 500),
"attack_type"
] = "brute_force"


df.loc[
(df["attack_type"] == "normal") &
(df["src_event_count"] >= 6) &
(df["unique_dst_count"] == 1) &
(df["unique_ports_per_src_ip"] == 1) &
(df["avg_bytes_per_event"] <= 300),
"attack_type"
] = "beaconing"

df = df[df["attack_type"].isin(["beaconing","brute_force","port_scan","network_scan","data_exfiltration","normal"])]

df[["src_ip","dst_ip","port","bytes","attack_type"]]


In [ ]:
reason_map = {
"port_scan": [
"Many unique ports contacted by the same source IP",
"Low average bytes per connection"
],
"network_scan": [
"Many unique destination IPs contacted",
"Low bytes per event",
"Multiple events from the same source"
],
"brute_force": [
"Repeated attempts to a single destination",
"Moderate traffic size per event"
],
"beaconing": [
"Frequent small periodic connections",
"Same destination and port"
],
"data_exfiltration": [
"Extremely large data transfer",
"High total bytes sent",
"Limited destination spread"
]
}

df["reasons"] = ""
df["reasons"] = df["attack_type"].map(reason_map)

df[df["attack_type"] != "normal"][["src_ip","src_event_count","bytes_sum","avg_bytes_per_event","max_bytes_per_event","unique_ports_per_src_ip","unique_dst_count","attack_type","reasons"]].drop_duplicates(subset="src_ip")


In [ ]:
df["attack_type"].value_counts()

In [ ]:
attack_summary = (df[df["attack_type"] != "normal"].groupby(["src_ip", "attack_type"]).agg(attack_events=("attack_type", "size"),total_bytes=("bytes", "sum")).sort_values("attack_events", ascending=False))

attack_summary

## Supervised baseline

Features are the per-source aggregates from above. Labels are the heuristic `attack_type` values, so this step checks whether a tree model can replay those rules — not whether it generalizes to new malware or new networks.

In [ ]:
X = df[["src_event_count", "bytes_sum", "avg_bytes_per_event", "max_bytes_per_event", "unique_ports_per_src_ip", "unique_dst_count"]]
y = df["attack_type"]



In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
%pip install xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1)

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()

In [ ]:
y_train_encoded = label_encoder.fit_transform(y_train)

In [ ]:
label_encoder.classes_

In [ ]:
y_test_encoded = label_encoder.transform(y_test)
y_test_encoded


In [ ]:
model.fit(X_train, y_train_encoded)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
y_pred_labels = label_encoder.inverse_transform(y_pred)

In [ ]:
y_pred_labels

In [ ]:
comparison = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred_labels
})

comparison

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test_encoded, y_pred)

print(accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_encoded, y_pred)

cm

In [ ]:
model.feature_importances_

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

features = X.columns
importance = model.feature_importances_

plt.figure(figsize=(10, 6))
plt.barh(features, importance)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importance")
plt.show()